# Overview

This notebook assembles a single master results file, `algorithm_results_cdrift.csv`, that combines concept-drift-detection results from several algorithms across the CDrift benchmark datasets (Ceravolo, Ostovar, Bose), so they can all be evaluated together in a downstream notebook ("Adam's notebook").

It does this in two stages, each following the same pattern: load a new algorithm's raw results, reformat them to match the master file's schema, backfill their ground-truth "Actual Changepoints" (which the raw results are missing) by looking them up from the `Zheng DBSCAN` rows for the same `Log`, then append them to the master file.

**Stage 1 - Add the "Bernold" (Version Clustering) results:**
- Starts from `algorithm_results_cdrift_original.csv`, the baseline file already containing results for established algorithms (Zheng DBSCAN, Process Graph Metrics, Earth Mover's Distance, LCDD, etc.).
- Loads the Bernold algorithm's per-dataset outputs from `datasets/cdrift/{Ceravolo,Ostovar,Bose}/Annotations/algorithm_results.csv` and concatenates them into one dataframe.
- Since these outputs don't include ground-truth changepoints, groups the baseline's `Zheng DBSCAN` rows by `Log` to recover the true changepoints and log source per log, then merges that onto the Bernold rows (dropping any log with no match).
- Appends the enriched Bernold rows to the baseline and writes the result to `algorithm_results_cdrift.csv`.

**Stage 2 - Add the "CV4CDD-4D" results:**
- Loads `res_cvdd_cdrift.csv`, a separate algorithm's raw output, and renames/reformats its columns (log name, duration, algorithm name, window size).
- Parses `Detected Changepoints` out of its tuple-string format (e.g. `"('518', '518')"`) into a flat, sorted list of integers.
- Reloads the just-updated `algorithm_results_cdrift.csv` and again uses its `Zheng DBSCAN` rows (grouped by `Log`) to backfill ground-truth changepoints for CV4CDD-4D, dropping logs with no match.
- Appends these rows to the master file and overwrites `algorithm_results_cdrift.csv`.

**Result:** `algorithm_results_cdrift.csv` ends up containing detection results from all algorithms (the original baselines, plus Bernold and CV4CDD-4D) in one consistent schema, ready for computing evaluation metrics (F1-score, average lag, etc.) in the downstream notebook.

Note: both backfill steps depend on a `Zheng DBSCAN` row existing for every `Log` - if it's ever missing for a log, that log is silently dropped rather than raising an error.

# Merge Version Clustering (Bernold) results with baseline CDrift algorithms, backfilling ground-truth changepoints from Zheng DBSCAN

In [1]:
import pandas as pd
import ast

# Read a CSV file
df_all_results = pd.read_csv("algorithm_results_cdrift_original.csv")

df_Ceravolo = pd.read_csv("datasets/cdrift/Ceravolo/Annotations/algorithm_results.csv")
df_Ostovar = pd.read_csv("datasets/cdrift/Ostovar/Annotations/algorithm_results.csv")
df_Bose = pd.read_csv("datasets/cdrift/Bose/Annotations/algorithm_results.csv")

df_combined = pd.concat([ df_Ceravolo ,df_Ostovar,df_Bose], ignore_index=True)
df_combined

,Algorithm,Log Source,Log,Detected Changepoints,Actual Changepoints for Log,Duration (Seconds),Duration,Window Size
0,Bernold,Ceravolo,sudden_trace_noise0_1000_cb,[494],[481],2.440087,00:00:02,75
1,Bernold,Ceravolo,sudden_trace_noise0_1000_cd,[],[481],2.051169,00:00:02,75
2,Bernold,Ceravolo,sudden_trace_noise0_1000_cf,[481],[481],2.320495,00:00:02,75
3,Bernold,Ceravolo,sudden_trace_noise0_1000_cp,[481],[481],2.607830,00:00:02,75
4,Bernold,Ceravolo,sudden_trace_noise0_1000_IOR,[481],[481],2.212932,00:00:02,75
...,...,...,...,...,...,...,...,...
296,Bernold,Ostovar,Nested_RIO_output_RIO_5,"[897, 1912]","[937, 1951]",11.609869,00:00:11,75
297,Bernold,Ostovar,Nested_ROI_output_ROI,"[895, 1913]","[937, 1950]",10.150690,00:00:10,75
298,Bernold,Ostovar,Nested_ROI_output_ROI_2,"[895, 1913]","[937, 1950]",9.686742,00:00:09,75
299,Bernold,Ostovar,Nested_ROI_output_ROI_5,"[895, 1913]","[937, 1950]",6.933439,00:00:06,75


In [2]:
df_all_results["Actual Changepoints for Log"] = df_all_results["Actual Changepoints for Log"].apply(ast.literal_eval)

df_temp = df_all_results[df_all_results['Algorithm'] == "Zheng DBSCAN"]
grouped = df_temp.groupby('Log').agg(
    Actual_Changepoints=('Actual Changepoints for Log', lambda series: sorted(set(x for lst in series for x in lst))),
    Log_Source=('Log Source', 'first')  # take the first Log Source for each Log
).reset_index()

grouped

,Log,Actual_Changepoints,Log_Source
0,Atomic_ConditionalMove_output_ConditionalMove,"[999, 1999]",Ostovar
1,Atomic_ConditionalMove_output_ConditionalMove_2,"[999, 1999]",Ostovar
2,Atomic_ConditionalMove_output_ConditionalMove_5,"[999, 1999]",Ostovar
3,Atomic_ConditionalRemoval_output_ConditionalRe...,"[999, 1999]",Ostovar
4,Atomic_ConditionalRemoval_output_ConditionalRe...,"[999, 1999]",Ostovar
...,...,...,...
146,sudden_trace_noise5_1000_pl,[499],Ceravolo
147,sudden_trace_noise5_1000_pm,[499],Ceravolo
148,sudden_trace_noise5_1000_re,[499],Ceravolo
149,sudden_trace_noise5_1000_rp,[499],Ceravolo


In [3]:
df_combined = df_combined.drop(columns=['Actual Changepoints for Log','Log Source'])

df_combined = df_combined.merge(grouped, on="Log", how="left")
df_combined['Actual_Changepoints'] = df_combined['Actual_Changepoints'].apply(lambda x: x if isinstance(x, list) else [])
df_combined = df_combined.dropna(subset=['Log_Source'])
df_combined = df_combined.rename(columns={'Log_Source': 'Log Source','Actual_Changepoints':'Actual Changepoints for Log'})
df_combined

,Algorithm,Log,Detected Changepoints,Duration (Seconds),Duration,Window Size,Actual Changepoints for Log,Log Source
0,Bernold,sudden_trace_noise0_1000_cb,[494],2.440087,00:00:02,75,[499],Ceravolo
1,Bernold,sudden_trace_noise0_1000_cd,[],2.051169,00:00:02,75,[499],Ceravolo
2,Bernold,sudden_trace_noise0_1000_cf,[481],2.320495,00:00:02,75,[499],Ceravolo
3,Bernold,sudden_trace_noise0_1000_cp,[481],2.607830,00:00:02,75,[499],Ceravolo
4,Bernold,sudden_trace_noise0_1000_IOR,[481],2.212932,00:00:02,75,[499],Ceravolo
...,...,...,...,...,...,...,...,...
296,Bernold,Nested_RIO_output_RIO_5,"[897, 1912]",11.609869,00:00:11,75,"[999, 1999]",Ostovar
297,Bernold,Nested_ROI_output_ROI,"[895, 1913]",10.150690,00:00:10,75,"[999, 1999]",Ostovar
298,Bernold,Nested_ROI_output_ROI_2,"[895, 1913]",9.686742,00:00:09,75,"[999, 1999]",Ostovar
299,Bernold,Nested_ROI_output_ROI_5,"[895, 1913]",6.933439,00:00:06,75,"[999, 1999]",Ostovar


In [4]:
df_all_results_updated = pd.concat([df_all_results, df_combined], ignore_index=True)
df_all_results_updated.to_csv("algorithm_results_cdrift.csv", index=False)

# Reformat CV4CDD-4D results and merge them into the combined results file, again backfilling ground-truth changepoints from Zheng DBSCAN

In [5]:
import pandas as pd

file_path = "res_cvdd_cdrift.csv" # IMPORTANT: Change to the where you have the file with the results from cvdd if you run it by yourself
df_cv4cdd = pd.read_csv(file_path)
df_cv4cdd

,Unnamed: 0,Detected Changepoints,Detected Drift Types,Prediction Confidence,time
0,sudden_trace_noise0_1000_re,"[('518', '518')]",['sudden'],[0.9029],4.041260
1,sudden_trace_noise10_100_cb,"[('50', '50')]",['sudden'],[0.5398],0.845845
2,sudden_trace_noise10_100_cd,"[('43', '67')]",['gradual'],[0.5594],0.845542
3,sudden_trace_noise20_100_lp,[],[],[],0.857207
4,sudden_trace_noise20_100_pl,[],[],[],0.872562
...,...,...,...,...,...
296,sudden_trace_noise10_100_lp,[],[],[],0.865158
297,sudden_trace_noise0_100_RIO,[],[],[],0.855350
298,sudden_trace_noise0_100_rp,"[('51', '51')]",['sudden'],[0.7368],0.859191
299,sudden_trace_noise5_1000_sw,"[('518', '518')]",['sudden'],[0.9207],1.520438


In [6]:
import time
import ast

df_cv4cdd = df_cv4cdd.rename(columns={'Unnamed: 0': 'Log','time':'Duration (Seconds)'})
df_cv4cdd["Algorithm"] = "CV4CDD-4D"
df_cv4cdd["Window Size"] = 70
df_cv4cdd['Duration'] = df_cv4cdd['Duration (Seconds)'].apply(lambda x: time.strftime('%H:%M:%S', time.gmtime(x)))

df_cv4cdd['Detected Changepoints'] = df_cv4cdd['Detected Changepoints'].apply(
    lambda s: sorted({int(x) for a, b in ast.literal_eval(s) for x in (a, b)})
)

df_cv4cdd

,Log,Detected Changepoints,Detected Drift Types,Prediction Confidence,Duration (Seconds),Algorithm,Window Size,Duration
0,sudden_trace_noise0_1000_re,[518],['sudden'],[0.9029],4.041260,CV4CDD-4D,70,00:00:04
1,sudden_trace_noise10_100_cb,[50],['sudden'],[0.5398],0.845845,CV4CDD-4D,70,00:00:00
2,sudden_trace_noise10_100_cd,"[43, 67]",['gradual'],[0.5594],0.845542,CV4CDD-4D,70,00:00:00
3,sudden_trace_noise20_100_lp,[],[],[],0.857207,CV4CDD-4D,70,00:00:00
4,sudden_trace_noise20_100_pl,[],[],[],0.872562,CV4CDD-4D,70,00:00:00
...,...,...,...,...,...,...,...,...
296,sudden_trace_noise10_100_lp,[],[],[],0.865158,CV4CDD-4D,70,00:00:00
297,sudden_trace_noise0_100_RIO,[],[],[],0.855350,CV4CDD-4D,70,00:00:00
298,sudden_trace_noise0_100_rp,[51],['sudden'],[0.7368],0.859191,CV4CDD-4D,70,00:00:00
299,sudden_trace_noise5_1000_sw,[518],['sudden'],[0.9207],1.520438,CV4CDD-4D,70,00:00:01


# Reload the combined results file (now including Bernold) before merging in CV4CDD-4D

In [7]:
# Path to your CSV file
file_path = "algorithm_results_cdrift.csv"

# Read the CSV into a DataFrame
df_all_results = pd.read_csv(file_path)

# Show the first few rows
df_all_results

,Algorithm,Log Source,Log,P-Value,Min Adaptive Window,Max Adaptive Window,Detected Changepoints,Actual Changepoints for Log,F1-Score,Average Lag,...,Duration (Seconds),Seconds per Case,Window Size,SW Step Size,Complete-Window Size,Detection-Window Size,Stable Period,ADWIN Step Size,MRID,Epsilon
0,Process Graph Metrics,Ostovar,Atomic_SerialMove_output_SerialMove_5,0.05,400.0,600.0,[994],"[999, 1999]",0.666667,5.0,...,41.920538,0.013978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Earth Mover's Distance,Ostovar,Atomic_Loop_output_Loop_2,NaN,NaN,NaN,"[956, 1932]","[999, 1999]",1.000000,55.0,...,1818.616341,0.606408,500.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,LCDD,Ceravolo,sudden_trace_noise20_1000_IOR,NaN,NaN,NaN,[447],[499],1.000000,52.0,...,0.017174,0.000017,NaN,NaN,400.0,400.0,50.0,NaN,NaN,NaN
3,Earth Mover's Distance,Bose,bose_log,NaN,NaN,NaN,"[1194, 2394, 3600, 4802]","[1199, 2399, 3599, 4799]",1.000000,3.5,...,639.391199,0.106565,600.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,LCDD,Ceravolo,sudden_trace_noise10_1000_IRO,NaN,NaN,NaN,[447],[499],1.000000,52.0,...,0.018100,0.000018,NaN,NaN,200.0,200.0,150.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32611,Bernold,Ostovar,Nested_RIO_output_RIO_5,NaN,NaN,NaN,"[897, 1912]","[999, 1999]",NaN,NaN,...,11.609869,NaN,75.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32612,Bernold,Ostovar,Nested_ROI_output_ROI,NaN,NaN,NaN,"[895, 1913]","[999, 1999]",NaN,NaN,...,10.150690,NaN,75.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32613,Bernold,Ostovar,Nested_ROI_output_ROI_2,NaN,NaN,NaN,"[895, 1913]","[999, 1999]",NaN,NaN,...,9.686742,NaN,75.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32614,Bernold,Ostovar,Nested_ROI_output_ROI_5,NaN,NaN,NaN,"[895, 1913]","[999, 1999]",NaN,NaN,...,6.933439,NaN,75.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_all_results["Actual Changepoints for Log"] = df_all_results["Actual Changepoints for Log"].apply(ast.literal_eval)

df_temp = df_all_results[df_all_results['Algorithm'] == "Zheng DBSCAN"]
grouped = df_temp.groupby('Log').agg(
    Actual_Changepoints=('Actual Changepoints for Log', lambda series: sorted(set(x for lst in series for x in lst))),
    Log_Source=('Log Source', 'first')  # take the first Log Source for each Log
).reset_index()

grouped

,Log,Actual_Changepoints,Log_Source
0,Atomic_ConditionalMove_output_ConditionalMove,"[999, 1999]",Ostovar
1,Atomic_ConditionalMove_output_ConditionalMove_2,"[999, 1999]",Ostovar
2,Atomic_ConditionalMove_output_ConditionalMove_5,"[999, 1999]",Ostovar
3,Atomic_ConditionalRemoval_output_ConditionalRe...,"[999, 1999]",Ostovar
4,Atomic_ConditionalRemoval_output_ConditionalRe...,"[999, 1999]",Ostovar
...,...,...,...
146,sudden_trace_noise5_1000_pl,[499],Ceravolo
147,sudden_trace_noise5_1000_pm,[499],Ceravolo
148,sudden_trace_noise5_1000_re,[499],Ceravolo
149,sudden_trace_noise5_1000_rp,[499],Ceravolo


In [9]:
df_cv4cdd = df_cv4cdd.merge(grouped, on="Log", how="left")
df_cv4cdd['Actual Changepoints for Log'] = df_cv4cdd['Actual_Changepoints'].apply(lambda x: x if isinstance(x, list) else [])
df_cv4cdd

,Log,Detected Changepoints,Detected Drift Types,Prediction Confidence,Duration (Seconds),Algorithm,Window Size,Duration,Actual_Changepoints,Log_Source,Actual Changepoints for Log
0,sudden_trace_noise0_1000_re,[518],['sudden'],[0.9029],4.041260,CV4CDD-4D,70,00:00:04,[499],Ceravolo,[499]
1,sudden_trace_noise10_100_cb,[50],['sudden'],[0.5398],0.845845,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
2,sudden_trace_noise10_100_cd,"[43, 67]",['gradual'],[0.5594],0.845542,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
3,sudden_trace_noise20_100_lp,[],[],[],0.857207,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
4,sudden_trace_noise20_100_pl,[],[],[],0.872562,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
...,...,...,...,...,...,...,...,...,...,...,...
296,sudden_trace_noise10_100_lp,[],[],[],0.865158,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
297,sudden_trace_noise0_100_RIO,[],[],[],0.855350,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
298,sudden_trace_noise0_100_rp,[51],['sudden'],[0.7368],0.859191,CV4CDD-4D,70,00:00:00,NaN,NaN,[]
299,sudden_trace_noise5_1000_sw,[518],['sudden'],[0.9207],1.520438,CV4CDD-4D,70,00:00:01,[499],Ceravolo,[499]


In [10]:
# Drop Nan from lgo source
df_cv4cdd = df_cv4cdd.dropna(subset=['Log_Source'])
df_cv4cdd

,Log,Detected Changepoints,Detected Drift Types,Prediction Confidence,Duration (Seconds),Algorithm,Window Size,Duration,Actual_Changepoints,Log_Source,Actual Changepoints for Log
0,sudden_trace_noise0_1000_re,[518],['sudden'],[0.9029],4.041260,CV4CDD-4D,70,00:00:04,[499],Ceravolo,[499]
5,sudden_trace_noise15_1000_IRO,[504],['sudden'],[0.9119],1.558415,CV4CDD-4D,70,00:00:01,[499],Ceravolo,[499]
9,Nested_ROI_output_ROI,"[967, 1976]","['sudden', 'sudden']",[0.9397 0.8636],6.910077,CV4CDD-4D,70,00:00:06,"[999, 1999]",Ostovar,"[999, 1999]"
12,Atomic_Frequency_output_Frequency_5,"[882, 967, 2898]","['sudden', 'recurring']",[0.7794 0.5725],6.069482,CV4CDD-4D,70,00:00:06,"[999, 1999]",Ostovar,"[999, 1999]"
14,Nested_RIO_output_RIO_2,"[966, 1933, 1975, 2899]","['sudden', 'sudden', 'incremental']",[0.878 0.8434 0.5205],8.789609,CV4CDD-4D,70,00:00:08,"[999, 1999]",Ostovar,"[999, 1999]"
...,...,...,...,...,...,...,...,...,...,...,...
289,Composite_IOR_output_IOR_5,"[884, 967, 1974, 2898]","['sudden', 'sudden', 'recurring']",[0.939 0.8573 0.5018],7.314633,CV4CDD-4D,70,00:00:07,"[999, 1999]",Ostovar,"[999, 1999]"
291,sudden_trace_noise15_1000_cp,[504],['sudden'],[0.9355],1.505981,CV4CDD-4D,70,00:00:01,[499],Ceravolo,[499]
292,sudden_trace_noise0_1000_RIO,[504],['sudden'],[0.5859],1.546037,CV4CDD-4D,70,00:00:01,[499],Ceravolo,[499]
299,sudden_trace_noise5_1000_sw,[518],['sudden'],[0.9207],1.520438,CV4CDD-4D,70,00:00:01,[499],Ceravolo,[499]


In [11]:
df_cv4cdd = df_cv4cdd.drop(columns=['Detected Drift Types', 'Prediction Confidence','Actual_Changepoints'])
df_cv4cdd = df_cv4cdd.rename(columns={'Log_Source': 'Log Source'})
df_cv4cdd

,Log,Detected Changepoints,Duration (Seconds),Algorithm,Window Size,Duration,Log Source,Actual Changepoints for Log
0,sudden_trace_noise0_1000_re,[518],4.041260,CV4CDD-4D,70,00:00:04,Ceravolo,[499]
5,sudden_trace_noise15_1000_IRO,[504],1.558415,CV4CDD-4D,70,00:00:01,Ceravolo,[499]
9,Nested_ROI_output_ROI,"[967, 1976]",6.910077,CV4CDD-4D,70,00:00:06,Ostovar,"[999, 1999]"
12,Atomic_Frequency_output_Frequency_5,"[882, 967, 2898]",6.069482,CV4CDD-4D,70,00:00:06,Ostovar,"[999, 1999]"
14,Nested_RIO_output_RIO_2,"[966, 1933, 1975, 2899]",8.789609,CV4CDD-4D,70,00:00:08,Ostovar,"[999, 1999]"
...,...,...,...,...,...,...,...,...
289,Composite_IOR_output_IOR_5,"[884, 967, 1974, 2898]",7.314633,CV4CDD-4D,70,00:00:07,Ostovar,"[999, 1999]"
291,sudden_trace_noise15_1000_cp,[504],1.505981,CV4CDD-4D,70,00:00:01,Ceravolo,[499]
292,sudden_trace_noise0_1000_RIO,[504],1.546037,CV4CDD-4D,70,00:00:01,Ceravolo,[499]
299,sudden_trace_noise5_1000_sw,[518],1.520438,CV4CDD-4D,70,00:00:01,Ceravolo,[499]


In [12]:
df_all_results_updated = pd.concat([df_all_results, df_cv4cdd], ignore_index=True)
df_all_results_updated

,Algorithm,Log Source,Log,P-Value,Min Adaptive Window,Max Adaptive Window,Detected Changepoints,Actual Changepoints for Log,F1-Score,Average Lag,...,Duration (Seconds),Seconds per Case,Window Size,SW Step Size,Complete-Window Size,Detection-Window Size,Stable Period,ADWIN Step Size,MRID,Epsilon
0,Process Graph Metrics,Ostovar,Atomic_SerialMove_output_SerialMove_5,0.05,400.0,600.0,[994],"[999, 1999]",0.666667,5.0,...,41.920538,0.013978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Earth Mover's Distance,Ostovar,Atomic_Loop_output_Loop_2,NaN,NaN,NaN,"[956, 1932]","[999, 1999]",1.000000,55.0,...,1818.616341,0.606408,500.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,LCDD,Ceravolo,sudden_trace_noise20_1000_IOR,NaN,NaN,NaN,[447],[499],1.000000,52.0,...,0.017174,0.000017,NaN,NaN,400.0,400.0,50.0,NaN,NaN,NaN
3,Earth Mover's Distance,Bose,bose_log,NaN,NaN,NaN,"[1194, 2394, 3600, 4802]","[1199, 2399, 3599, 4799]",1.000000,3.5,...,639.391199,0.106565,600.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,LCDD,Ceravolo,sudden_trace_noise10_1000_IRO,NaN,NaN,NaN,[447],[499],1.000000,52.0,...,0.018100,0.000018,NaN,NaN,200.0,200.0,150.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32762,CV4CDD-4D,Ostovar,Composite_IOR_output_IOR_5,NaN,NaN,NaN,"[884, 967, 1974, 2898]","[999, 1999]",NaN,NaN,...,7.314633,NaN,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32763,CV4CDD-4D,Ceravolo,sudden_trace_noise15_1000_cp,NaN,NaN,NaN,[504],[499],NaN,NaN,...,1.505981,NaN,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32764,CV4CDD-4D,Ceravolo,sudden_trace_noise0_1000_RIO,NaN,NaN,NaN,[504],[499],NaN,NaN,...,1.546037,NaN,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32765,CV4CDD-4D,Ceravolo,sudden_trace_noise5_1000_sw,NaN,NaN,NaN,[518],[499],NaN,NaN,...,1.520438,NaN,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_all_results_updated.to_csv('algorithm_results_cdrift.csv', index=False, encoding='utf-8')